In [1]:
# install dependencies
!pip install rdkit xgboost


In [2]:
## import the xlsx with compound SMILES strings
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools

# Load only the specified columns from the Excel file
df = pd.read_excel('AID_720635_cleaned.xlsx', usecols=['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_OUTCOME'])

PandasTools.AddMoleculeColumnToFrame(df, 'PUBCHEM_EXT_DATASOURCE_SMILES', 'Molecule')

print(df[['PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_SCORE', 'Molecule']].head())

               PUBCHEM_EXT_DATASOURCE_SMILES  PUBCHEM_ACTIVITY_SCORE  \
0                                    CC(CN)O                      10   
1  C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                      83   
2                                   C(CCl)Cl                       0   
3                      C1=CC(=C(C=C1Cl)Cl)Cl                      10   
4                                   C(C=O)Cl                      10   

                                        Molecule  
0  <rdkit.Chem.rdchem.Mol object at 0x303ca3a00>  
1  <rdkit.Chem.rdchem.Mol object at 0x303ca3a70>  
2  <rdkit.Chem.rdchem.Mol object at 0x303ca3990>  
3  <rdkit.Chem.rdchem.Mol object at 0x303ca38b0>  
4  <rdkit.Chem.rdchem.Mol object at 0x303ca3680>  


[13:56:45] WARNING: not removing hydrogen atom without neighbors
[13:56:45] WARNING: not removing hydrogen atom without neighbors
[13:56:45] WARNING: not removing hydrogen atom without neighbors


In [3]:
from rdkit.Chem import AllChem

# Function to generate Morgan fingerprint (ECFP4) as a bit vector
def get_morgan_fingerprint(mol):
    if mol is not None:
        return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048) # Radius 2, 2048 bits
    return None

# Apply the function to the 'Molecule' column to create a new 'Morgan_Fingerprint' column
df['Morgan_Fingerprint'] = df['Molecule'].apply(get_morgan_fingerprint)

print(df[['PUBCHEM_EXT_DATASOURCE_SMILES', 'Molecule', 'Morgan_Fingerprint']].head())

               PUBCHEM_EXT_DATASOURCE_SMILES  \
0                                    CC(CN)O   
1  C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl   
2                                   C(CCl)Cl   
3                      C1=CC(=C(C=C1Cl)Cl)Cl   
4                                   C(C=O)Cl   

                                        Molecule  \
0  <rdkit.Chem.rdchem.Mol object at 0x303ca3a00>   
1  <rdkit.Chem.rdchem.Mol object at 0x303ca3a70>   
2  <rdkit.Chem.rdchem.Mol object at 0x303ca3990>   
3  <rdkit.Chem.rdchem.Mol object at 0x303ca38b0>   
4  <rdkit.Chem.rdchem.Mol object at 0x303ca3680>   

                                  Morgan_Fingerprint  
0  [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  


[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerator
[13:57:02] DEPRECATION WARNING: please use MorganGenerat

# Task
Create a predictive model to predict `PUBCHEM_ACTIVITY_SCORE` using Morgan fingerprints as features. This involves handling missing fingerprints, converting fingerprints to a suitable format, splitting the data into training and testing sets, and then training and evaluating three different regression models: Support Vector Machine (SVM), Random Forest, and XGBoost. Finally, compare the performance of these models using relevant metrics.

## Prepare data for ML

### Subtask:
Handle any rows where Morgan_Fingerprint is None and convert fingerprints to a suitable format for scikit-learn. Define features (X) and target (y).


**Reasoning**:
First, I will remove rows where the 'Morgan_Fingerprint' is None to ensure all entries have valid fingerprints. Then, I will convert the RDKit BitVector fingerprints into a numerical NumPy array suitable for machine learning, and finally, define the features (X) and the target variable (y).



In [4]:
import numpy as np

# 1. Remove rows with None in 'Morgan_Fingerprint'
df_cleaned = df.dropna(subset=['Morgan_Fingerprint'])

# 2. Convert Morgan_Fingerprint from RDKit BitVector to a NumPy array
X = np.array([np.array(fp) for fp in df_cleaned['Morgan_Fingerprint']])

# 3. Define the target variable y
y = df_cleaned['PUBCHEM_ACTIVITY_SCORE']

print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")
print(f"First 5 rows of X:\n{X[:5]}")
print(f"First 5 values of y:\n{y[:5]}")

Shape of X (features): (8099, 2048)
Shape of y (target): (8099,)
First 5 rows of X:
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
First 5 values of y:
0    10
1    83
2     0
3    10
4    10
Name: PUBCHEM_ACTIVITY_SCORE, dtype: int64


# Task
## Split data into train, validation, and test sets

### Subtask:
Split the prepared data into 80% training, 10% validation, and 10% testing sets using `train_test_split`, ensuring reproducibility.

### Reasoning:
To achieve an 80% training, 10% validation, and 10% testing split, I will first split the data into a temporary training/validation set (90%) and a final test set (10%). Then, I will split the temporary training/validation set into the final training set (8/9 of the temporary set, which is 80% of the original data) and the validation set (1/9 of the temporary set, which is 10% of the original data). The `random_state` will be used for reproducibility.

## Split data into train, validation, and test sets

### Subtask:
Split the prepared data into 80% training, 10% validation, and 10% testing sets using train_test_split, ensuring reproducibility.


**Reasoning**:
To achieve the 80/10/10 train/validation/test split, I will first split the data into a 90% temporary training/validation set and a 10% test set. Then, I will split the temporary set into 8/9 for training and 1/9 for validation to get the final 80% training and 10% validation sets. Finally, I will print the shapes of all sets to verify the proportions.



In [5]:
from sklearn.model_selection import train_test_split

# 1. First split: 90% temporary train/validation, 10% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# 2. Second split: From the 90% temporary set, split into 80% train and 10% validation
# (8/9 of X_temp, y_temp for train, and 1/9 for validation)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=42)

# 3. Print the shapes of the resulting sets
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (6479, 2048)
Shape of y_train: (6479,)
Shape of X_val: (810, 2048)
Shape of y_val: (810,)
Shape of X_test: (810, 2048)
Shape of y_test: (810,)


# Task
Train, tune, and evaluate an SVM regressor model using the prepared training data. Identify optimal hyperparameters by evaluating different combinations on the validation set, and then assess the best model's final performance on the test set.

## Train, tune, and evaluate SVM model

### Subtask:
Iterate through different SVM hyperparameters, train on the training set, evaluate performance on the validation set to select the best model, and then assess its final performance on the test set.


**Reasoning**:
I need to import the necessary libraries for SVR modeling and evaluation, define the hyperparameter grid for tuning, and initialize lists to store the performance metrics and hyperparameters as instructed.



In [6]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import itertools
import numpy as np

# 2. Define a dictionary, param_grid, specifying the hyperparameters to tune
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 1],
    'epsilon': [0.1, 0.01]
}

# 3. Initialize empty lists to store the validation R-squared, MAE, and RMSE scores,
# as well as the corresponding hyperparameters and test set metrics.

val_r2_scores = []
val_mae_scores = []
val_rmse_scores = []

test_r2_scores = []
test_mae_scores = []
test_rmse_scores = []

hyperparameter_combinations = []

# Variables to store the best model's metrics and hyperparameters
best_val_r2 = -float('inf')
best_hyperparams = None
best_test_metrics = {}

print("Libraries imported, param_grid defined, and lists initialized.")

Libraries imported, param_grid defined, and lists initialized.


**Reasoning**:
I need to iterate through all possible hyperparameter combinations, train an SVR model for each combination, evaluate its performance on both validation and test sets, and keep track of the best performing model based on the validation R-squared score, as per instructions #4, #5, #6, and #7.



In [7]:
for C, gamma, epsilon in itertools.product(param_grid['C'], param_grid['gamma'], param_grid['epsilon']):
    hyperparams = {'C': C, 'gamma': gamma, 'epsilon': epsilon}
    hyperparameter_combinations.append(hyperparams)

    # Initialize and train the SVR model
    svr = SVR(C=C, gamma=gamma, epsilon=epsilon)
    svr.fit(X_train, y_train)

    # Predict on the validation set
    y_val_pred = svr.predict(X_val)

    # Calculate validation metrics
    r2_val = r2_score(y_val, y_val_pred)
    mae_val = mean_absolute_error(y_val, y_val_pred)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

    val_r2_scores.append(r2_val)
    val_mae_scores.append(mae_val)
    val_rmse_scores.append(rmse_val)

    # Predict on the test set
    y_test_pred = svr.predict(X_test)

    # Calculate test metrics
    r2_test = r2_score(y_test, y_test_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    test_r2_scores.append(r2_test)
    test_mae_scores.append(mae_test)
    test_rmse_scores.append(rmse_test)

    # Check if this is the best model based on validation R-squared
    if r2_val > best_val_r2:
        best_val_r2 = r2_val
        best_hyperparams = hyperparams
        best_test_metrics = {
            'r2': r2_test,
            'mae': mae_test,
            'rmse': rmse_test
        }

print("Hyperparameter tuning complete. Best model found.")

# Print the optimal hyperparameters and their corresponding metrics
print("\nOptimal SVR Hyperparameters:", best_hyperparams)
print(f"Best Validation R-squared: {best_val_r2:.4f}")
print("\nTest Set Performance (with best model):")
print(f"R-squared: {best_test_metrics['r2']:.4f}")
print(f"MAE: {best_test_metrics['mae']:.4f}")
print(f"RMSE: {best_test_metrics['rmse']:.4f}")

Hyperparameter tuning complete. Best model found.

Optimal SVR Hyperparameters: {'C': 10, 'gamma': 'scale', 'epsilon': 0.1}
Best Validation R-squared: 0.1958

Test Set Performance (with best model):
R-squared: 0.1988
MAE: 12.8189
RMSE: 21.3766


## Train, tune, and evaluate Random Forest model

### Subtask:
Iterate through different Random Forest hyperparameters, train on the training set, evaluate performance on the validation set to select the best model, and then assess its final performance on the test set.


**Reasoning**:
I need to import the necessary libraries for RandomForestRegressor modeling and evaluation, define the hyperparameter grid for tuning, and initialize lists to store the performance metrics and hyperparameters as instructed.



In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import itertools
import numpy as np

# 2. Define a dictionary, param_grid_rf, specifying the hyperparameters to tune
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# 3. Initialize empty lists to store the validation R-squared, MAE, and RMSE scores,
# as well as the corresponding hyperparameters and test set metrics.

val_r2_scores_rf = []
val_mae_scores_rf = []
val_rmse_scores_rf = []

test_r2_scores_rf = []
test_mae_scores_rf = []
test_rmse_scores_rf = []

hyperparameter_combinations_rf = []

# Variables to store the best model's metrics and hyperparameters
best_val_r2_rf = -float('inf')
best_hyperparams_rf = None
best_test_metrics_rf = {}

print("Libraries imported, param_grid_rf defined, and lists initialized for Random Forest.")

Libraries imported, param_grid_rf defined, and lists initialized for Random Forest.


# Task
## Train, tune, and evaluate Random Forest model: Iterate through different Random Forest hyperparameters, train on the training set, evaluate performance on the validation set to select the best model, and then assess its final performance on the test set.

## Train, tune, and evaluate Random Forest model

### Subtask:
Iterate through different Random Forest hyperparameters, train on the training set, evaluate performance on the validation set to select the best model, and then assess its final performance on the test set.


**Reasoning**:
I need to iterate through all possible hyperparameter combinations for Random Forest, train a model for each combination, evaluate its performance on both validation and test sets, and keep track of the best performing model based on the validation R-squared score, as per the instructions.



In [9]:
for n_estimators, max_depth, min_samples_split in itertools.product(param_grid_rf['n_estimators'], param_grid_rf['max_depth'], param_grid_rf['min_samples_split']):
    hyperparams_rf = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split
    }
    hyperparameter_combinations_rf.append(hyperparams_rf)

    # Initialize and train the RandomForestRegressor model
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,  # for reproducibility
        n_jobs=-1 # Use all available cores
    )
    rf_model.fit(X_train, y_train)

    # Predict on the validation set
    y_val_pred_rf = rf_model.predict(X_val)

    # Calculate validation metrics
    r2_val_rf = r2_score(y_val, y_val_pred_rf)
    mae_val_rf = mean_absolute_error(y_val, y_val_pred_rf)
    rmse_val_rf = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))

    val_r2_scores_rf.append(r2_val_rf)
    val_mae_scores_rf.append(mae_val_rf)
    val_rmse_scores_rf.append(rmse_val_rf)

    # Predict on the test set
    y_test_pred_rf = rf_model.predict(X_test)

    # Calculate test metrics
    r2_test_rf = r2_score(y_test, y_test_pred_rf)
    mae_test_rf = mean_absolute_error(y_test, y_test_pred_rf)
    rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

    test_r2_scores_rf.append(r2_test_rf)
    test_mae_scores_rf.append(mae_test_rf)
    test_rmse_scores_rf.append(rmse_test_rf)

    # Check if this is the best model based on validation R-squared
    if r2_val_rf > best_val_r2_rf:
        best_val_r2_rf = r2_val_rf
        best_hyperparams_rf = hyperparams_rf
        best_test_metrics_rf = {
            'r2': r2_test_rf,
            'mae': mae_test_rf,
            'rmse': rmse_test_rf
        }

print("Random Forest Hyperparameter tuning complete. Best model found.")

# Print the optimal hyperparameters and their corresponding metrics
print("\nOptimal Random Forest Hyperparameters:", best_hyperparams_rf)
print(f"Best Validation R-squared: {best_val_r2_rf:.4f}")
print("\nTest Set Performance (with best Random Forest model):")
print(f"R-squared: {best_test_metrics_rf['r2']:.4f}")
print(f"MAE: {best_test_metrics_rf['mae']:.4f}")
print(f"RMSE: {best_test_metrics_rf['rmse']:.4f}")

Random Forest Hyperparameter tuning complete. Best model found.

Optimal Random Forest Hyperparameters: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2}
Best Validation R-squared: 0.3336

Test Set Performance (with best Random Forest model):
R-squared: 0.3216
MAE: 12.9801
RMSE: 19.6703


# Task
Train, tune, and evaluate an XGBoost regressor model, and then compare its performance with the SVM and Random Forest models.

## Train, tune, and evaluate XGBoost model

### Subtask:
Iterate through different XGBoost hyperparameters, train on the training set, evaluate performance on the validation set to select the best model, and then assess its final performance on the test set.


**Reasoning**:
I need to import the necessary libraries for XGBoostRegressor modeling and evaluation, define the hyperparameter grid for tuning, and initialize lists to store the performance metrics and hyperparameters as instructed.



In [10]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import itertools
import numpy as np

# 2. Define a dictionary, param_grid_xgb, specifying the hyperparameters to tune
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1]
}

# 3. Initialize empty lists to store the validation R-squared, MAE, and RMSE scores,
# as well as the corresponding hyperparameters and test set metrics.

val_r2_scores_xgb = []
val_mae_scores_xgb = []
val_rmse_scores_xgb = []

test_r2_scores_xgb = []
test_mae_scores_xgb = []
test_rmse_scores_xgb = []

hyperparameter_combinations_xgb = []

# Variables to store the best model's metrics and hyperparameters
best_val_r2_xgb = -float('inf')
best_hyperparams_xgb = None
best_test_metrics_xgb = {}

print("Libraries imported, param_grid_xgb defined, and lists initialized for XGBoost.")

Libraries imported, param_grid_xgb defined, and lists initialized for XGBoost.


**Reasoning**:
I need to iterate through all possible hyperparameter combinations for XGBoost, train a model for each combination, evaluate its performance on both validation and test sets, and keep track of the best performing model based on the validation R-squared score, as per the instructions.



In [11]:
for n_estimators, max_depth, learning_rate in itertools.product(param_grid_xgb['n_estimators'], param_grid_xgb['max_depth'], param_grid_xgb['learning_rate']):
    hyperparams_xgb = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate
    }
    hyperparameter_combinations_xgb.append(hyperparams_xgb)

    # Initialize and train the XGBRegressor model
    xgb_model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42, # for reproducibility
        n_jobs=-1 # Use all available cores
    )
    xgb_model.fit(X_train, y_train)

    # Predict on the validation set
    y_val_pred_xgb = xgb_model.predict(X_val)

    # Calculate validation metrics
    r2_val_xgb = r2_score(y_val, y_val_pred_xgb)
    mae_val_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
    rmse_val_xgb = np.sqrt(mean_squared_error(y_val, y_val_pred_xgb))

    val_r2_scores_xgb.append(r2_val_xgb)
    val_mae_scores_xgb.append(mae_val_xgb)
    val_rmse_scores_xgb.append(rmse_val_xgb)

    # Predict on the test set
    y_test_pred_xgb = xgb_model.predict(X_test)

    # Calculate test metrics
    r2_test_xgb = r2_score(y_test, y_test_pred_xgb)
    mae_test_xgb = mean_absolute_error(y_test, y_test_pred_xgb)
    rmse_test_xgb = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))

    test_r2_scores_xgb.append(r2_test_xgb)
    test_mae_scores_xgb.append(mae_test_xgb)
    test_rmse_scores_xgb.append(rmse_test_xgb)

    # Check if this is the best model based on validation R-squared
    if r2_val_xgb > best_val_r2_xgb:
        best_val_r2_xgb = r2_val_xgb
        best_hyperparams_xgb = hyperparams_xgb
        best_test_metrics_xgb = {
            'r2': r2_test_xgb,
            'mae': mae_test_xgb,
            'rmse': rmse_test_xgb
        }

print("XGBoost Hyperparameter tuning complete. Best model found.")

# Print the optimal hyperparameters and their corresponding metrics
print("\nOptimal XGBoost Hyperparameters:", best_hyperparams_xgb)
print(f"Best Validation R-squared: {best_val_r2_xgb:.4f}")
print("\nTest Set Performance (with best XGBoost model):")
print(f"R-squared: {best_test_metrics_xgb['r2']:.4f}")
print(f"MAE: {best_test_metrics_xgb['mae']:.4f}")
print(f"RMSE: {best_test_metrics_xgb['rmse']:.4f}")

XGBoost Hyperparameter tuning complete. Best model found.

Optimal XGBoost Hyperparameters: {'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1}
Best Validation R-squared: 0.3969

Test Set Performance (with best XGBoost model):
R-squared: 0.3460
MAE: 12.5133
RMSE: 19.3134


# Task
Compare the performance of the SVM, Random Forest, and XGBoost models based on their test set R-squared, MAE, and RMSE metrics.

## Compare model performance

### Subtask:
Compare the performance metrics (e.g., R-squared, MAE, RMSE) of the SVM, Random Forest, and XGBoost models.


In [12]:
import pandas as pd

print("\n--- Model Performance Comparison (Test Set) ---")

# SVM Performance
print("\nSVM Model (Best Hyperparameters):", best_hyperparams)
print(f"  R-squared: {best_test_metrics['r2']:.4f}")
print(f"  MAE: {best_test_metrics['mae']:.4f}")
print(f"  RMSE: {best_test_metrics['rmse']:.4f}")

# Random Forest Performance
# Note: This assumes best_test_metrics_rf has been populated by the RF tuning process
if best_hyperparams_rf:
    print("\nRandom Forest Model (Best Hyperparameters):", best_hyperparams_rf)
    print(f"  R-squared: {best_test_metrics_rf['r2']:.4f}")
    print(f"  MAE: {best_test_metrics_rf['mae']:.4f}")
    print(f"  RMSE: {best_test_metrics_rf['rmse']:.4f}")
else:
    print("\nRandom Forest Model: Metrics not available (tuning not completed).")

# XGBoost Performance
# Note: This assumes best_test_metrics_xgb has been populated by the XGBoost tuning process
if best_hyperparams_xgb:
    print("\nXGBoost Model (Best Hyperparameters):", best_hyperparams_xgb)
    print(f"  R-squared: {best_test_metrics_xgb['r2']:.4f}")
    print(f"  MAE: {best_test_metrics_xgb['mae']:.4f}")
    print(f"  RMSE: {best_test_metrics_xgb['rmse']:.4f}")
else:
    print("\nXGBoost Model: Metrics not available (tuning not completed).")

# Create a DataFrame for a more structured comparison if all metrics are available
comparison_data = []
comparison_data.append({'Model': 'SVM', 'R-squared': best_test_metrics['r2'], 'MAE': best_test_metrics['mae'], 'RMSE': best_test_metrics['rmse']})

if best_hyperparams_rf:
    comparison_data.append({'Model': 'Random Forest', 'R-squared': best_test_metrics_rf['r2'], 'MAE': best_test_metrics_rf['mae'], 'RMSE': best_test_metrics_rf['rmse']})
else:
    comparison_data.append({'Model': 'Random Forest', 'R-squared': None, 'MAE': None, 'RMSE': None})

if best_hyperparams_xgb:
    comparison_data.append({'Model': 'XGBoost', 'R-squared': best_test_metrics_xgb['r2'], 'MAE': best_test_metrics_xgb['mae'], 'RMSE': best_test_metrics_xgb['rmse']})
else:
    comparison_data.append({'Model': 'XGBoost', 'R-squared': None, 'MAE': None, 'RMSE': None})

comparison_df = pd.DataFrame(comparison_data)

print("\n--- Detailed Model Comparison (DataFrame) ---")
display(comparison_df.round(4))



--- Model Performance Comparison (Test Set) ---

SVM Model (Best Hyperparameters): {'C': 10, 'gamma': 'scale', 'epsilon': 0.1}
  R-squared: 0.1988
  MAE: 12.8189
  RMSE: 21.3766

Random Forest Model (Best Hyperparameters): {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2}
  R-squared: 0.3216
  MAE: 12.9801
  RMSE: 19.6703

XGBoost Model (Best Hyperparameters): {'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1}
  R-squared: 0.3460
  MAE: 12.5133
  RMSE: 19.3134

--- Detailed Model Comparison (DataFrame) ---


,Model,R-squared,MAE,RMSE
0,SVM,0.1988,12.8189,21.3766
1,Random Forest,0.3216,12.9801,19.6703
2,XGBoost,0.3460,12.5133,19.3134


# Task
Convert the RDKit molecule objects in the DataFrame into a graph data format suitable for Graph Neural Networks (GNNs), including defining atom and bond features.

## Convert Molecules to Graph Data

### Subtask:
Transform the RDKit molecule objects in the DataFrame into a graph data format suitable for GNNs, including defining atom and bond features.


**Reasoning**:
I need to import the necessary libraries, define helper functions for extracting atom and bond features from RDKit molecules, and then iterate through the DataFrame to convert each molecule into a `torch_geometric.data.Data` object.



In [13]:
!pip install torch torch-geometric


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.4 MB/s eta 0:00:00


In [14]:
import torch
from torch_geometric.data import Data
from rdkit.Chem import AllChem

# 1. Define atom_features function
def atom_features(atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()), # Convert HybridizationType to int
        int(atom.GetIsAromatic())
    ], dtype=torch.float)

# 2. Define bond_features function
def bond_features(bond):
    return torch.tensor([
        int(bond.GetBondType()), # Convert BondType to int
        int(bond.GetStereo()),   # Convert BondStereo to int
        int(bond.GetIsConjugated())
    ], dtype=torch.float)

# 3. Initialize an empty list to store graph objects
data_list = []

# 4. Iterate through each row of the df_cleaned DataFrame
for index, row in df_cleaned.iterrows():
    mol = row['Molecule']
    if mol is None: # Skip rows where molecule could not be parsed
        continue

    # Extract atom features (x)
    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()])

    # Extract edge indices (edge_index) and bond features (edge_attr)
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index.append([i, j])
        edge_attr.append(bond_features(bond))
        # For undirected graph, add reverse edge
        edge_index.append([j, i])
        edge_attr.append(bond_features(bond))

    if not edge_index:
        # Handle molecules with no bonds (e.g., single atoms or malformed SMILES)
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, len(bond_features(Chem.MolFromSmiles('C-C').GetBondWithIdx(0)))), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.stack(edge_attr)

    # Define the target variable y
    y = torch.tensor([row['PUBCHEM_ACTIVITY_SCORE']], dtype=torch.float)

    # Create a torch_geometric.data.Data object
    graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data_list.append(graph_data)

print(f"Generated {len(data_list)} graph objects.")
# Print the first graph object to inspect its structure and features
if data_list:
    print("\nFirst graph object in data_list:")
    print(data_list[0])
    print("\nFeatures of first atom:")
    print(data_list[0].x[0])
    if data_list[0].edge_index.shape[1] > 0:
        print("\nFeatures of first bond (from first edge):")
        print(data_list[0].edge_attr[0])

Generated 8099 graph objects.

First graph object in data_list:
Data(x=[5, 5], edge_index=[2, 8], edge_attr=[8, 3], y=[1])

Features of first atom:
tensor([6., 1., 0., 4., 0.])

Features of first bond (from first edge):
tensor([1., 0., 0.])


# Task
Split the newly prepared graph datasets into 80% training, 10% validation, and 10% testing sets, mirroring the previous data split proportions for consistency.

## Split Graph Data

### Subtask:
Split the newly prepared graph datasets into 80% training, 10% validation, and 10% testing sets, mirroring the previous data split proportions for consistency.


**Reasoning**:
To split the graph datasets into 80% training, 10% validation, and 10% testing sets, I will use `train_test_split` twice, first to separate the test set and then to split the remaining data into training and validation sets, ensuring reproducibility with `random_state=42`.



In [15]:
from sklearn.model_selection import train_test_split

# 1. First split: 90% temporary train/validation, 10% test
data_temp, data_test = train_test_split(data_list, test_size=0.1, random_state=42)

# 2. Second split: From the 90% temporary set, split into 80% train and 10% validation
# (8/9 of data_temp for train, and 1/9 for validation)
data_train, data_val = train_test_split(data_temp, test_size=(1/9), random_state=42)

# 3. Print the number of graphs in the resulting sets
print(f"Number of graphs in training set: {len(data_train)}")
print(f"Number of graphs in validation set: {len(data_val)}")
print(f"Number of graphs in test set: {len(data_test)}")

Number of graphs in training set: 6479
Number of graphs in validation set: 810
Number of graphs in test set: 810


# Task
Define and implement various Graph Neural Network architectures (e.g., GCN, GAT, GIN, GraphSAGE) suitable for regression tasks, with appropriate layers for processing molecular graphs and outputting a regression score.

## Define GNN Model Architecture

### Subtask:
Design and implement various Graph Neural Network architectures (e.g., GCN, GAT, GIN, GraphSAGE) suitable for regression tasks, with appropriate layers for processing molecular graphs and outputting a regression score.


**Reasoning**:
I need to import the necessary modules from `torch` and `torch_geometric.nn` as instructed to prepare for defining the GNN model architectures.



In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv, global_add_pool, MessagePassing
from torch.nn import Linear

print("Necessary modules imported for GNN model definition.")

Necessary modules imported for GNN model definition.


**Reasoning**:
As instructed, I will first define a base `GNNModel` class with common initialization and a forward pass structure, and then implement `GCNModel`, `GATModel`, `GINModel`, and `GraphSAGENet` as subclasses, each utilizing specific GNN layers and implementing their unique forward logic for regression tasks.



In [17]:
class GNNModel(nn.Module):
    def __init__(self, num_features, hidden_channels, num_classes=1):
        super(GNNModel, self).__init__()
        torch.manual_seed(12345)
        self.initial_lin = Linear(num_features, hidden_channels)
        self.dropout = nn.Dropout(0.5)
        self.pooling = global_add_pool
        self.lin_out = Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        # Initial feature projection
        x = self.initial_lin(x)
        x = F.relu(x)
        x = self.dropout(x)

        # GNN layer(s) will be applied by subclasses
        # (placeholder for now)
        x = x # This will be overridden by subclasses

        # Global pooling
        x = self.pooling(x, batch)

        # Final regression head
        x = self.lin_out(x)
        return x

# GCN Model Implementation
class GCNModel(GNNModel):
    def __init__(self, num_features, hidden_channels, num_classes=1):
        super(GCNModel, self).__init__(num_features, hidden_channels, num_classes)
        self.conv1 = GCNConv(hidden_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index, batch):
        x = self.initial_lin(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.pooling(x, batch)
        x = self.lin_out(x)
        return x

# GAT Model Implementation
class GATModel(GNNModel):
    def __init__(self, num_features, hidden_channels, num_classes=1, heads=4):
        super(GATModel, self).__init__(num_features, hidden_channels, num_classes)
        # GATConv output channels are hidden_channels * heads if concat=True
        # We set concat=False to maintain hidden_channels output dimension after first layer
        # or adjust hidden_channels to be divisible by heads if concat=True
        self.conv1 = GATConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.5)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.5)

    def forward(self, x, edge_index, batch):
        x = self.initial_lin(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.pooling(x, batch)
        x = self.lin_out(x)
        return x

# GIN Model Implementation
class GINModel(GNNModel):
    def __init__(self, num_features, hidden_channels, num_classes=1):
        super(GINModel, self).__init__(num_features, hidden_channels, num_classes)
        # Define MLPs for GINConv layers
        nn1 = nn.Sequential(Linear(hidden_channels, hidden_channels), nn.ReLU(), Linear(hidden_channels, hidden_channels))
        nn2 = nn.Sequential(Linear(hidden_channels, hidden_channels), nn.ReLU(), Linear(hidden_channels, hidden_channels))

        self.conv1 = GINConv(nn1)
        self.conv2 = GINConv(nn2)

    def forward(self, x, edge_index, batch):
        x = self.initial_lin(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.pooling(x, batch)
        x = self.lin_out(x)
        return x

# GraphSAGE Model Implementation
class GraphSAGENet(GNNModel):
    def __init__(self, num_features, hidden_channels, num_classes=1):
        super(GraphSAGENet, self).__init__(num_features, hidden_channels, num_classes)
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index, batch):
        x = self.initial_lin(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.pooling(x, batch)
        x = self.lin_out(x)
        return x

print("GNN model architectures (GNNModel, GCNModel, GATModel, GINModel, GraphSAGENet) defined.")

GNN model architectures (GNNModel, GCNModel, GATModel, GINModel, GraphSAGENet) defined.


## Train and Tune GNN Model

### Subtask:
Train the defined GNN models on the graph training data. This will include defining a loss function, optimizer, and iterating through epochs. Hyperparameters will be tuned using the validation set for each architecture.


**Reasoning**:
I need to prepare the data for training by creating PyTorch DataLoader instances for the training, validation, and test datasets. I will also define the loss function and optimizer that will be used for training the GNN models.



In [18]:
from torch_geometric.loader import DataLoader

# Get the number of features from the first graph object
num_features = data_train[0].x.shape[1]
hidden_channels = 64 # Define hidden channels for GNNs

# Initialize the GNN models
gcn_model = GCNModel(num_features, hidden_channels)
gat_model = GATModel(num_features, hidden_channels)
gin_model = GINModel(num_features, hidden_channels)
graphsage_model = GraphSAGENet(num_features, hidden_channels)

# Define the loss function (Mean Squared Error for regression)
loss_fn = torch.nn.MSELoss()

# Define optimizers for each model
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=0.001)
optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=0.001)
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=0.001)
optimizer_graphsage = torch.optim.Adam(graphsage_model.parameters(), lr=0.001)

# Create PyTorch DataLoader instances for the datasets
batch_size = 32 # Can be adjusted
train_loader = DataLoader(data_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(data_val, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(data_test, batch_size=batch_size, shuffle=False)

print(f"Number of features: {num_features}")
print(f"Hidden channels: {hidden_channels}")
print("GNN models initialized.")
print("Loss function (MSELoss) and Adam optimizers defined for each model.")
print(f"DataLoaders created with batch size: {batch_size}.")


Number of features: 5
Hidden channels: 64
GNN models initialized.
Loss function (MSELoss) and Adam optimizers defined for each model.
DataLoaders created with batch size: 32.


**Reasoning**:
I need to implement a training loop for each GNN model. This involves defining a training function to perform forward and backward passes, and an evaluation function to assess performance on the validation set after each epoch. I will then iterate through epochs for each model, track the best performing model based on validation RMSE, and store its state.



In [19]:
from torch_geometric.data import Batch
from tqdm.notebook import tqdm

def train_gnn_model(model, optimizer, loader, loss_fn, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = loss_fn(out.squeeze(), data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate_gnn_model(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            loss = loss_fn(out.squeeze(), data.y)
            total_loss += loss.item() * data.num_graphs
            all_preds.append(out.squeeze().cpu())
            all_targets.append(data.y.cpu())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))

    return avg_loss, r2, mae, rmse


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dictionary to store best models and their metrics
best_gnn_models = {}

# --- Train and Evaluate GCN Model ---
print("\n--- Training GCN Model ---")
gcn_model = gcn_model.to(device)
best_val_rmse_gcn = float('inf')
for epoch in tqdm(range(1, 51), desc="GCN Training"): # Run for 50 epochs
    train_loss = train_gnn_model(gcn_model, optimizer_gcn, train_loader, loss_fn, device)
    val_loss, val_r2, val_mae, val_rmse = evaluate_gnn_model(gcn_model, val_loader, loss_fn, device)
    # print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val R2: {val_r2:.4f}, Val MAE: {val_mae:.4f}, Val RMSE: {val_rmse:.4f}')

    if val_rmse < best_val_rmse_gcn:
        best_val_rmse_gcn = val_rmse
        best_gnn_models['GCN'] = {
            'model_state_dict': gcn_model.state_dict(),
            'hyperparams': {'hidden_channels': hidden_channels}, # Assuming simple hyperparams for now
            'val_r2': val_r2,
            'val_mae': val_mae,
            'val_rmse': val_rmse
        }

# --- Train and Evaluate GAT Model ---
print("\n--- Training GAT Model ---")
gat_model = gat_model.to(device)
best_val_rmse_gat = float('inf')
for epoch in tqdm(range(1, 51), desc="GAT Training"): # Run for 50 epochs
    train_loss = train_gnn_model(gat_model, optimizer_gat, train_loader, loss_fn, device)
    val_loss, val_r2, val_mae, val_rmse = evaluate_gnn_model(gat_model, val_loader, loss_fn, device)
    # print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val R2: {val_r2:.4f}, Val MAE: {val_mae:.4f}, Val RMSE: {val_rmse:.4f}')

    if val_rmse < best_val_rmse_gat:
        best_val_rmse_gat = val_rmse
        best_gnn_models['GAT'] = {
            'model_state_dict': gat_model.state_dict(),
            'hyperparams': {'hidden_channels': hidden_channels, 'heads': gat_model.conv1.heads}, # Add heads to hyperparams
            'val_r2': val_r2,
            'val_mae': val_mae,
            'val_rmse': val_rmse
        }

# --- Train and Evaluate GIN Model ---
print("\n--- Training GIN Model ---")
gin_model = gin_model.to(device)
best_val_rmse_gin = float('inf')
for epoch in tqdm(range(1, 51), desc="GIN Training"): # Run for 50 epochs
    train_loss = train_gnn_model(gin_model, optimizer_gin, train_loader, loss_fn, device)
    val_loss, val_r2, val_mae, val_rmse = evaluate_gnn_model(gin_model, val_loader, loss_fn, device)
    # print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val R2: {val_r2:.4f}, Val MAE: {val_mae:.4f}, Val RMSE: {val_rmse:.4f}')

    if val_rmse < best_val_rmse_gin:
        best_val_rmse_gin = val_rmse
        best_gnn_models['GIN'] = {
            'model_state_dict': gin_model.state_dict(),
            'hyperparams': {'hidden_channels': hidden_channels},
            'val_r2': val_r2,
            'val_mae': val_mae,
            'val_rmse': val_rmse
        }

# --- Train and Evaluate GraphSAGE Model ---
print("\n--- Training GraphSAGE Model ---")
graphsage_model = graphsage_model.to(device)
best_val_rmse_graphsage = float('inf')
for epoch in tqdm(range(1, 51), desc="GraphSAGE Training"): # Run for 50 epochs
    train_loss = train_gnn_model(graphsage_model, optimizer_graphsage, train_loader, loss_fn, device)
    val_loss, val_r2, val_mae, val_rmse = evaluate_gnn_model(graphsage_model, val_loader, loss_fn, device)
    # print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val R2: {val_r2:.4f}, Val MAE: {val_mae:.4f}, Val RMSE: {val_rmse:.4f}')

    if val_rmse < best_val_rmse_graphsage:
        best_val_rmse_graphsage = val_rmse
        best_gnn_models['GraphSAGE'] = {
            'model_state_dict': graphsage_model.state_dict(),
            'hyperparams': {'hidden_channels': hidden_channels},
            'val_r2': val_r2,
            'val_mae': val_mae,
            'val_rmse': val_rmse
        }

print("\n--- GNN Model Training Complete ---")
for model_name, metrics in best_gnn_models.items():
    print(f"Best {model_name} Model on Validation Set:")
    print(f"  R2: {metrics['val_r2']:.4f}")
    print(f"  MAE: {metrics['val_mae']:.4f}")
    print(f"  RMSE: {metrics['val_rmse']:.4f}")
    print(f"  Hyperparameters: {metrics['hyperparams']}")



--- Training GCN Model ---


GCN Training:   0%|          | 0/50 [00:00<?, ?it/s]


--- Training GAT Model ---


GAT Training:   0%|          | 0/50 [00:00<?, ?it/s]


--- Training GIN Model ---


GIN Training:   0%|          | 0/50 [00:00<?, ?it/s]


--- Training GraphSAGE Model ---


GraphSAGE Training:   0%|          | 0/50 [00:00<?, ?it/s]


--- GNN Model Training Complete ---
Best GCN Model on Validation Set:
  R2: 0.1488
  MAE: 15.1523
  RMSE: 22.3515
  Hyperparameters: {'hidden_channels': 64}
Best GAT Model on Validation Set:
  R2: 0.1461
  MAE: 14.7699
  RMSE: 22.3880
  Hyperparameters: {'hidden_channels': 64, 'heads': 4}
Best GIN Model on Validation Set:
  R2: 0.1730
  MAE: 14.7313
  RMSE: 22.0318
  Hyperparameters: {'hidden_channels': 64}
Best GraphSAGE Model on Validation Set:
  R2: 0.1778
  MAE: 14.5008
  RMSE: 21.9674
  Hyperparameters: {'hidden_channels': 64}


**Reasoning**:
Now that all GNN models have been trained and the best performing models on the validation set identified, I need to evaluate these best GNN models on the unseen test set to get their final performance metrics. This is the next logical step in the 'Train and Tune GNN Model' subtask, as per instructions #6.



In [20]:
print("\n--- Evaluating Best GNN Models on Test Set ---")

test_gnn_results = {}

for model_name, metrics in best_gnn_models.items():
    print(f"\nEvaluating {model_name} Model...")
    # Re-initialize model to load best state_dict
    if model_name == 'GCN':
        model_instance = GCNModel(num_features, hidden_channels)
    elif model_name == 'GAT':
        model_instance = GATModel(num_features, hidden_channels, heads=metrics['hyperparams']['heads'])
    elif model_name == 'GIN':
        model_instance = GINModel(num_features, hidden_channels)
    elif model_name == 'GraphSAGE':
        model_instance = GraphSAGENet(num_features, hidden_channels)
    else:
        continue

    model_instance.load_state_dict(metrics['model_state_dict'])
    model_instance = model_instance.to(device)

    # Evaluate on test set
    test_loss, test_r2, test_mae, test_rmse = evaluate_gnn_model(model_instance, test_loader, loss_fn, device)

    test_gnn_results[model_name] = {
        'r2': test_r2,
        'mae': test_mae,
        'rmse': test_rmse
    }

    print(f"Test R-squared: {test_r2:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Test RMSE: {test_rmse:.4f}")

print("\n--- All GNN Models Test Set Evaluation Complete ---")



--- Evaluating Best GNN Models on Test Set ---

Evaluating GCN Model...
Test R-squared: 0.1885
Test MAE: 14.9822
Test RMSE: 21.5147

Evaluating GAT Model...
Test R-squared: 0.1672
Test MAE: 13.9751
Test RMSE: 21.7945

Evaluating GIN Model...
Test R-squared: 0.2153
Test MAE: 14.5537
Test RMSE: 21.1556

Evaluating GraphSAGE Model...
Test R-squared: 0.1927
Test MAE: 13.9769
Test RMSE: 21.4586

--- All GNN Models Test Set Evaluation Complete ---


## Compare model performance

### Subtask:
Compare the performance metrics (e.g., R-squared, MAE, RMSE) of the SVM, Random Forest, XGBoost, and GNN models.


**Reasoning**:
I will consolidate and display the performance metrics for SVM, Random Forest, XGBoost, and all GNN models (GCN, GAT, GIN, GraphSAGE) from their respective dictionaries. This will involve printing the individual metrics and then creating a comprehensive pandas DataFrame for a structured comparison, handling cases where Random Forest metrics might be missing.



In [21]:
import pandas as pd

print("\n--- Final Model Performance Comparison (Test Set) ---")

# SVM Performance
print("\nSVM Model (Best Hyperparameters):", best_hyperparams)
print(f"  R-squared: {best_test_metrics['r2']:.4f}")
print(f"  MAE: {best_test_metrics['mae']:.4f}")
print(f"  RMSE: {best_test_metrics['rmse']:.4f}")

# Random Forest Performance
if best_hyperparams_rf:
    print("\nRandom Forest Model (Best Hyperparameters):", best_hyperparams_rf)
    print(f"  R-squared: {best_test_metrics_rf['r2']:.4f}")
    print(f"  MAE: {best_test_metrics_rf['mae']:.4f}")
    print(f"  RMSE: {best_test_metrics_rf['rmse']:.4f}")
else:
    print("\nRandom Forest Model: Metrics not available (tuning not completed or failed).")

# XGBoost Performance
if best_hyperparams_xgb:
    print("\nXGBoost Model (Best Hyperparameters):", best_hyperparams_xgb)
    print(f"  R-squared: {best_test_metrics_xgb['r2']:.4f}")
    print(f"  MAE: {best_test_metrics_xgb['mae']:.4f}")
    print(f"  RMSE: {best_test_metrics_xgb['rmse']:.4f}")
else:
    print("\nXGBoost Model: Metrics not available (tuning not completed or failed).")

# GNN Models Performance
for model_name, metrics in test_gnn_results.items():
    print(f"\n{model_name} Model Performance (Test Set):")
    print(f"  R-squared: {metrics['r2']:.4f}")
    print(f"  MAE: {metrics['mae']:.4f}")
    print(f"  RMSE: {metrics['rmse']:.4f}")

# Create a DataFrame for a structured comparison
comparison_data = []

# Add SVM metrics
comparison_data.append({'Model': 'SVM', 'R-squared': best_test_metrics['r2'], 'MAE': best_test_metrics['mae'], 'RMSE': best_test_metrics['rmse']})

# Add Random Forest metrics
if best_hyperparams_rf:
    comparison_data.append({'Model': 'Random Forest', 'R-squared': best_test_metrics_rf['r2'], 'MAE': best_test_metrics_rf['mae'], 'RMSE': best_test_metrics_rf['rmse']})
else:
    comparison_data.append({'Model': 'Random Forest', 'R-squared': None, 'MAE': None, 'RMSE': None})

# Add XGBoost metrics
if best_hyperparams_xgb:
    comparison_data.append({'Model': 'XGBoost', 'R-squared': best_test_metrics_xgb['r2'], 'MAE': best_test_metrics_xgb['mae'], 'RMSE': best_test_metrics_xgb['rmse']})
else:
    comparison_data.append({'Model': 'XGBoost', 'R-squared': None, 'MAE': None, 'RMSE': None})

# Add GNN models metrics
for model_name, metrics in test_gnn_results.items():
    comparison_data.append({'Model': model_name, 'R-squared': metrics['r2'], 'MAE': metrics['mae'], 'RMSE': metrics['rmse']})

comparison_df_all = pd.DataFrame(comparison_data)

print("\n--- Detailed Model Comparison (DataFrame) ---")
display(comparison_df_all.round(4))



--- Final Model Performance Comparison (Test Set) ---

SVM Model (Best Hyperparameters): {'C': 10, 'gamma': 'scale', 'epsilon': 0.1}
  R-squared: 0.1988
  MAE: 12.8189
  RMSE: 21.3766

Random Forest Model (Best Hyperparameters): {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2}
  R-squared: 0.3216
  MAE: 12.9801
  RMSE: 19.6703

XGBoost Model (Best Hyperparameters): {'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1}
  R-squared: 0.3460
  MAE: 12.5133
  RMSE: 19.3134

GCN Model Performance (Test Set):
  R-squared: 0.1885
  MAE: 14.9822
  RMSE: 21.5147

GAT Model Performance (Test Set):
  R-squared: 0.1672
  MAE: 13.9751
  RMSE: 21.7945

GIN Model Performance (Test Set):
  R-squared: 0.2153
  MAE: 14.5537
  RMSE: 21.1556

GraphSAGE Model Performance (Test Set):
  R-squared: 0.1927
  MAE: 13.9769
  RMSE: 21.4586

--- Detailed Model Comparison (DataFrame) ---


,Model,R-squared,MAE,RMSE
0,SVM,0.1988,12.8189,21.3766
1,Random Forest,0.3216,12.9801,19.6703
2,XGBoost,0.3460,12.5133,19.3134
3,GCN,0.1885,14.9822,21.5147
4,GAT,0.1672,13.9751,21.7945
5,GIN,0.2153,14.5537,21.1556
6,GraphSAGE,0.1927,13.9769,21.4586
